# Test SAM-Med3D Feature Quality

This notebook evaluates whether SAM-Med3D features are discriminative for tumor classification by training a simple classification head on top of the 3D encodings.

Before running the full TabPFN/LoCalPFN pipeline, we want to check if the SAM-Med3D features capture useful information for benign vs malignant classification.

In [1]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from med3pipe.data.prepare import prepare_for_sam3d, split_validation, find_default_sam3d_root
from med3pipe.sam.core import load_labels_from_sheet
from med3pipe.training.classification_head import run_classification_head_experiment

c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [7]:
# Dataset configuration
DATASET = "gist"  # Change to 'lipo' or other dataset
CATEGORY = DATASET
CT_NAME = f"ct_{DATASET.upper()}"
DATASET_ROOT = PROJECT_ROOT / "data" / DATASET
DATASET_NAME = DATASET.upper()  # For filtering in sheet.csv
CASE_SUFFIX = "_CT"  # or "_MR" for MRI datasets

# Paths
SHEET_CSV = PROJECT_ROOT / "data" / "sheet.csv"  # or PROJECT_ROOT / "sheet.csv" for unified sheet
# Auto-detect SAM-Med3D checkpoint
SAM3D_CHECKPOINT = sam3d_root / "ckpt" / "sam_med3d_turbo.pth"
if not SAM3D_CHECKPOINT.exists():
    SAM3D_CHECKPOINT = sam3d_root / "ckpt" / "SAM-Med3D-turbo.pth"
if not SAM3D_CHECKPOINT.exists():
    print("⚠️  WARNING: No checkpoint found! Model will use random weights.")
    print(f"   Download from: https://huggingface.co/blueyo0/SAM-Med3D/blob/main/sam_med3d_turbo.pth")
    print(f"   Save to: {sam3d_root / 'ckpt' / 'sam_med3d_turbo.pth'}")
    SAM3D_CHECKPOINT = None
else:
    print(f"✅ Using checkpoint: {SAM3D_CHECKPOINT}")

# Training configuration
FREEZE_ENCODER = True  # Set to False to fine-tune encoder
NUM_EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.3
IMG_SIZE = 128

# Split configuration
SPLIT_RATIO = 0.8
SEED = 2025

# Output
OUTPUT_DIR = PROJECT_ROOT / "results" / "classification_head" / DATASET

print(f"Configuration:")
print(f"  Dataset: {DATASET}")
print(f"  Dataset root: {DATASET_ROOT}")
print(f"  Freeze encoder: {FREEZE_ENCODER}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Output: {OUTPUT_DIR}")

Configuration:
  Dataset: gist
  Dataset root: c:\Users\cahel\Desktop\Med3Tab-PFN\data\gist
  Freeze encoder: True
  Epochs: 3
  Output: c:\Users\cahel\Desktop\Med3Tab-PFN\results\classification_head\gist


## Step 1: Prepare Dataset

In [3]:
sam3d_root = find_default_sam3d_root()
print(f"SAM-Med3D root: {sam3d_root}")

prepared, paths = prepare_for_sam3d(
    dataset_root=DATASET_ROOT,
    sam3d_root=sam3d_root,
    category=CATEGORY,
    ct_name=CT_NAME,
    case_glob=None,
    max_cases=None,
)

print(f"\n✓ Prepared {prepared} cases")
print(f"  Train dir: {paths.train_root}")
print(f"  Val dir: {paths.val_root}")

SAM-Med3D root: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main
Prepared 25 cases ...
Prepared 50 cases ...
Prepared 75 cases ...
Prepared 100 cases ...
Prepared 125 cases ...
Prepared 150 cases ...
Prepared 175 cases ...
Prepared 200 cases ...
Prepared 225 cases ...
Done. Prepared 246 cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST

✓ Prepared 246 cases
  Train dir: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST
  Val dir: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST


## Step 2: Create Validation Split

In [4]:
split_validation(
    paths,
    split_ratio=SPLIT_RATIO,
    seed=SEED,
    copy=True,
)

print("✓ Validation split created")

Validation set copied to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST | Train: 196 | Val: 50
✓ Validation split created


## Step 3: Load Labels

In [8]:
df, lab_map = load_labels_from_sheet(
    sheet_csv=SHEET_CSV,
    dataset_name=DATASET_NAME,
    subject_col="Subject",
    label_col="Diagnosis_binary",
    case_suffix=CASE_SUFFIX,
)

print(f"✓ Loaded {len(lab_map)} labels")
print(f"\nClass distribution:")
print(df['label'].value_counts())
print(f"\nBenign: {(df['label'] == 0).sum()}")
print(f"Malignant: {(df['label'] == 1).sum()}")

✓ Loaded 246 labels

Class distribution:
label
1    125
0    121
Name: count, dtype: int64

Benign: 121
Malignant: 125


## Step 4: Run Classification Head Experiment

In [13]:
import importlib
import med3pipe.training.classification_head
importlib.reload(med3pipe.training.classification_head)
from med3pipe.training import run_classification_head_experiment
import importlib
import med3pipe.sam.core
import med3pipe.training.classification_head

importlib.reload(med3pipe.sam.core)
importlib.reload(med3pipe.training.classification_head)

from med3pipe.training import run_classification_head_experiment

In [14]:
results = run_classification_head_experiment(
    paths=paths,
    lab_map=lab_map,
    sam3d_root=sam3d_root,
    model_type="vit_b_ori",
    checkpoint=SAM3D_CHECKPOINT,
    img_size=IMG_SIZE,
    device=None,  # Auto-detect
    freeze_encoder=FREEZE_ENCODER,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    dropout=DROPOUT,
    num_workers=2,
    output_dir=OUTPUT_DIR,
)


SAM-Med3D Classification Head Experiment
Device: cpu
Model: vit_b_ori
Encoder frozen: True
Image size: 128
Batch size: 4
Epochs: 3
Learning rate: 0.001

[1/4] Building SAM-Med3D model...
[2/4] Adding classification head...
[INFO] SAM-Med3D encoder frozen. Training only classification head.
[3/4] Preparing dataloaders...
[INFO] Train samples: 246, Val samples: 50
[4/4] Starting training...

Starting training for 3 epochs

  Batch [20/62] Loss: 0.4576
  Batch [40/62] Loss: 0.7386
  Batch [60/62] Loss: 0.6347

Epoch [1/3] - Time: 66176.49s
Train Loss: 0.7195 | Train Acc: 0.4919
Val Loss:   0.6744 | Val Acc:   0.6200
Val AUC:    0.6899
LR:         0.000753
✓ Best model saved (AUC: 0.6899)

  Batch [20/62] Loss: 0.5594
  Batch [40/62] Loss: 0.7151
  Batch [60/62] Loss: 0.9324

Epoch [2/3] - Time: 3885.89s
Train Loss: 0.7043 | Train Acc: 0.5366
Val Loss:   0.6675 | Val Acc:   0.5400
Val AUC:    0.6575
LR:         0.000258

  Batch [20/62] Loss: 0.6593
  Batch [40/62] Loss: 0.6230
  Batch [6

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([scalar])` or the `torch.serialization.safe_globals([scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weig

## Results Analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot training curves
history = results['history']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy and AUC
axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['val_acc'], label='Val Acc', marker='s')
axes[1].plot(history['val_auc'], label='Val AUC', marker='^')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].set_title('Accuracy and AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Plot saved to {OUTPUT_DIR / 'training_curves.png'}")

In [ ]:
# Final metrics
metrics = results['final_metrics']

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"Best Epoch: {results['best_epoch']}")
print(f"Best Validation AUC: {results['best_auc']:.4f}")
print(f"Final Accuracy: {metrics.accuracy:.4f}")
print(f"Final AUC: {metrics.auc:.4f}")
print(f"\nConfusion Matrix:")
print(metrics.confusion_matrix)
print(f"\nClassification Report:")
print(metrics.classification_report)
print("="*60)

In [ ]:
# Plot confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=metrics.confusion_matrix,
    display_labels=["Benign", "Malignant"]
)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Confusion matrix saved to {OUTPUT_DIR / 'confusion_matrix.png'}")

In [ ]:
# ROC curve
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(metrics.targets, metrics.probabilities)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ ROC curve saved to {OUTPUT_DIR / 'roc_curve.png'}")

## Interpretation

**Good Features** (AUC > 0.7 with frozen encoder):
- SAM-Med3D learned meaningful representations
- Proceed with TabPFN/LoCalPFN pipeline
- Consider fine-tuning for better performance

**Poor Features** (AUC < 0.6 even with fine-tuning):
- SAM-Med3D features may not be suitable for this task
- Consider:
  - Different pretrained weights
  - Different feature extraction strategy
  - Alternative architectures
  - Check data quality and labels

**Moderate Features** (0.6 < AUC < 0.7):
- Features have some signal but limited
- TabPFN/LoCalPFN might help extract more from features
- Consider fine-tuning SAM-Med3D on your data